In [8]:
!pip install -q sentence-transformers fugashi ipadic

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split

# --- データ準備 ---
def load_lines(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

chuni_data = load_lines('chuni_sentences.txt')
normal_data = load_lines('normal_sentences.txt')

# 8:2に分割
train_chuni, test_chuni = train_test_split(chuni_data, test_size=0.2, random_state=42)
train_normal, test_normal = train_test_split(normal_data, test_size=0.2, random_state=42)

# 検証用データ（正解ラベル: 1=厨二, 0=一般）
test_sentences = test_chuni + test_normal
test_labels = [1] * len(test_chuni) + [0] * len(test_normal)
total_chuni_count = len(test_chuni) # 正解の総数（これを目指す）

model_names = [
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'intfloat/multilingual-e5-large',
    'intfloat/multilingual-e5-large-instruct',
    'intfloat/multilingual-e5-small',
    'Alibaba-NLP/gte-multilingual-base',
    'nomic-ai/nomic-embed-text-v2-moe',
    'minishlab/potion-multilingual-128M',
    'sonoisa/sentence-bert-base-ja-mean-tokens-v2',
    'sonoisa/sentence-bert-base-ja-mean-tokens',
    'pkshatech/GLuCoSE-base-ja-v2',
    'sentence-transformers/all-MiniLM-L6-v2',
]

print(f"=== ランキング精度対決（検証用データ総数: {len(test_sentences)}件 / 正解: {total_chuni_count}件） ===\n")

for model_name in model_names:
    model = SentenceTransformer(model_name)

    # 1. 厨二病ベクトル作成
    emb_train_chuni = model.encode(train_chuni)
    emb_train_normal = model.encode(train_normal)
    chuni_vec = np.mean(emb_train_chuni, axis=0) - np.mean(emb_train_normal, axis=0)

    # 2. 検証データのスコア計算とランク付け
    emb_test = model.encode(test_sentences)
    scores = util.cos_sim(emb_test, chuni_vec).numpy().flatten()

    # スコア順に並べ替え [(スコア, 正解ラベル), ...]
    ranking = sorted(zip(scores, test_labels), key=lambda x: x[0], reverse=True)

    # 3. 各順位での正解数をカウント
    top_10 = [label for _, label in ranking[:10]]
    top_20 = [label for _, label in ranking[:20]]
    # R-Precision: 正解データ数(26個)と同じ順位まで見たとき、何個正解があるか
    top_R  = [label for _, label in ranking[:total_chuni_count]]

    acc_10 = sum(top_10) / 10 * 100
    acc_20 = sum(top_20) / 20 * 100
    acc_R  = sum(top_R) / total_chuni_count * 100

    print(f"モデル: {model_name}")
    print(f"  - 上位10件の厨二病率: {acc_10:.1f}% ({sum(top_10)}/10)")
    print(f"  - 上位20件の厨二病率: {acc_20:.1f}% ({sum(top_20)}/20)")
    print(f"  ★ 総合力(R-Precision): {acc_R:.1f}% ({sum(top_R)}/{total_chuni_count})")

    # 最初に間違えた順位を探す
    first_mistake = next((i+1 for i, (_, label) in enumerate(ranking) if label == 0), "なし")
    print(f"  - 最初の誤検知（一般文）: {first_mistake}位")
    print("-" * 30)

=== ランキング精度対決（検証用データ総数: 52件 / 正解: 26件） ===



モデル: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - 上位10件の厨二病率: 100.0% (10/10)
  - 上位20件の厨二病率: 95.0% (19/20)
  ★ 総合力(R-Precision): 96.2% (25/26)
  - 最初の誤検知（一般文）: 18位
------------------------------
モデル: sonoisa/sentence-bert-base-ja-mean-tokens
  - 上位10件の厨二病率: 100.0% (10/10)
  - 上位20件の厨二病率: 100.0% (20/20)
  ★ 総合力(R-Precision): 92.3% (24/26)
  - 最初の誤検知（一般文）: 23位
------------------------------
モデル: pkshatech/GLuCoSE-base-ja-v2
  - 上位10件の厨二病率: 100.0% (10/10)
  - 上位20件の厨二病率: 100.0% (20/20)
  ★ 総合力(R-Precision): 100.0% (26/26)
  - 最初の誤検知（一般文）: 27位
------------------------------
モデル: sentence-transformers/all-MiniLM-L6-v2
  - 上位10件の厨二病率: 100.0% (10/10)
  - 上位20件の厨二病率: 100.0% (20/20)
  ★ 総合力(R-Precision): 100.0% (26/26)
  - 最初の誤検知（一般文）: 27位
------------------------------


In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split

# --- データ準備 ---
def load_lines(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

chuni_data = load_lines('chuni_sentences.txt')
normal_data = load_lines('normal_sentences.txt')

train_chuni, test_chuni = train_test_split(chuni_data, test_size=0.2, random_state=42)
train_normal, test_normal = train_test_split(normal_data, test_size=0.2, random_state=42)

model_names = [
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'intfloat/multilingual-e5-large',
    'sonoisa/sentence-bert-base-ja-mean-tokens',
    'pkshatech/GLuCoSE-base-ja-v2',
    'sentence-transformers/all-MiniLM-L6-v2',
]

print("=== 決戦：分離能力テスト ===")

for model_name in model_names:
    model = SentenceTransformer(model_name)

    # 1. ベクトル作成（学習データ）
    emb_train_chuni = model.encode(train_chuni)
    emb_train_normal = model.encode(train_normal)
    chuni_vec = np.mean(emb_train_chuni, axis=0) - np.mean(emb_train_normal, axis=0)

    # 2. テストデータのスコア計算
    # 厨二病データのスコア
    scores_chuni = util.cos_sim(model.encode(test_chuni), chuni_vec).numpy().flatten()
    # 一般データのスコア
    scores_normal = util.cos_sim(model.encode(test_normal), chuni_vec).numpy().flatten()

    # 3. 指標計算
    avg_chuni = scores_chuni.mean()
    avg_normal = scores_normal.mean()
    margin = avg_chuni - avg_normal  # この値が大きいほど優秀

    # 4. 「一般文章」なのに一番厨二病っぽかった文（誤検知リスク）
    worst_idx = np.argmax(scores_normal)
    worst_score = scores_normal[worst_idx]
    worst_text = test_normal[worst_idx]

    print(f"\nモデル: {model_name}")
    print(f"  - 厨二平均スコア: {avg_chuni:.4f}")
    print(f"  - 一般平均スコア: {avg_normal:.4f}")
    print(f"  - マージン（差）: {margin:.4f}  <-- これが最大のモデルが優勝！")
    print(f"  - 一般の中で最も厨二値が高かった文(スコア:{worst_score:.4f}):")
    print(f"    「{worst_text[:30]}...」")

=== 決戦：分離能力テスト ===

モデル: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - 厨二平均スコア: 0.5266
  - 一般平均スコア: 0.0057
  - マージン（差）: 0.5209  <-- これが最大のモデルが優勝！
  - 一般の中で最も厨二値が高かった文(スコア:0.4785):
    「この「もて」は、分けつすることを意味する「もてる」「もでる」...」



モデル: sonoisa/sentence-bert-base-ja-mean-tokens
  - 厨二平均スコア: 0.4979
  - 一般平均スコア: 0.1151
  - マージン（差）: 0.3828  <-- これが最大のモデルが優勝！
  - 一般の中で最も厨二値が高かった文(スコア:0.4063):
    「内裏・朝堂院の構造がそれまで見られなかった大規模で画期的な物...」

モデル: pkshatech/GLuCoSE-base-ja-v2
  - 厨二平均スコア: 0.3213
  - 一般平均スコア: -0.0667
  - マージン（差）: 0.3881  <-- これが最大のモデルが優勝！
  - 一般の中で最も厨二値が高かった文(スコア:0.0242):
    「この「もて」は、分けつすることを意味する「もてる」「もでる」...」

モデル: sentence-transformers/all-MiniLM-L6-v2
  - 厨二平均スコア: 0.2218
  - 一般平均スコア: -0.1140
  - マージン（差）: 0.3358  <-- これが最大のモデルが優勝！
  - 一般の中で最も厨二値が高かった文(スコア:0.0856):
    「ワケネギ（わけねぎ、分けねぎ、分け葱）とは、株分れ（分けつ）...」
